# scFoundation Two-Group MSE Analysis

This notebook scores two lung subsets with scFoundation, compares their masked-gene MSE distributions, and checks whether MSE or predictability track with sequencing depth, age, BMI, or smoking status.

Lower masked MSE means the cell is easier for the model to predict.

In [ ]:
import gc
import sys
from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.sparse import issparse


def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents, start / "ChaosScore", Path("/content/ChaosScore")]
    for candidate in candidates:
        if (candidate / "external" / "scFoundation" / "model").exists():
            return candidate
    raise FileNotFoundError("Could not find the ChaosScore repo root from the current working directory")


def clear_gpu_memory(*objects) -> None:
    for obj in objects:
        if obj is not None:
            del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


REPO_ROOT = find_repo_root(Path.cwd())
SCF_MODEL_DIR = REPO_ROOT / "external" / "scFoundation" / "model"
CKPT_PATH = SCF_MODEL_DIR / "models.ckpt"
AGE_CUTOFF = 30
GROUP_PATHS = {
    "healthy": REPO_ROOT / "data" / "processed" / "lung" / f"{AGE_CUTOFF}_non_smoking_normal_healthy" / "subset.h5ad",
    "risk_group": REPO_ROOT / "data" / "processed" / "lung" / f"older_than_{AGE_CUTOFF}_smoked_normal_risk_group" / "subset.h5ad",
}
COUNTS_SOURCE = "raw"
MAX_CELLS_PER_GROUP = 512  # set to None to score every cell; the risk group is much larger and slower
BATCH_SIZE = 1  # keep small on 8 GB GPUs
MASK_FRACTION = 0.30
RANDOM_SEED = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GENE_NAME_COLUMNS = ("gene_name", "feature_name")
SMOKING_CODE = {"never": 0, "former": 1, "active": 2}

print(f"REPO_ROOT={REPO_ROOT}")
print(f"DEVICE={DEVICE}")
print(pd.Series({name: str(path) for name, path in GROUP_PATHS.items()}))

In [ ]:
if not CKPT_PATH.exists():
    raise FileNotFoundError(f"Missing checkpoint: {CKPT_PATH}")

for group_name, group_path in GROUP_PATHS.items():
    if not group_path.exists():
        raise FileNotFoundError(f"Missing group file for {group_name}: {group_path}")

if str(SCF_MODEL_DIR) not in sys.path:
    sys.path.insert(0, str(SCF_MODEL_DIR))

from load import getEncoerDecoderData, load_model_frommmf, main_gene_selection

pd.Series({
    "checkpoint": str(CKPT_PATH),
    "max_cells_per_group": MAX_CELLS_PER_GROUP,
    "batch_size": BATCH_SIZE,
    "mask_fraction": MASK_FRACTION,
})

In [ ]:
gene_panel = pd.read_csv(SCF_MODEL_DIR / "OS_scRNA_gene_index.19264.tsv", sep="\t")
gene_list = gene_panel["gene_name"].astype(str).tolist()


def load_group_counts_df(
    adata_path: Path,
    *,
    counts_source: str,
    max_cells: int | None,
    random_seed: int,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    adata = ad.read_h5ad(adata_path, backed="r")
    try:
        if max_cells is None or adata.n_obs <= max_cells:
            selected_pos = np.arange(adata.n_obs)
        else:
            rng = np.random.default_rng(random_seed)
            selected_pos = np.sort(rng.choice(adata.n_obs, size=max_cells, replace=False))

        obs = adata.obs.iloc[selected_pos].copy()
        if counts_source == "raw":
            if adata.raw is None:
                raise ValueError(f"counts_source='raw' but adata.raw is missing in {adata_path}")
            matrix = adata.raw.X[selected_pos, :]
            var = adata.raw.var.copy()
        else:
            matrix = adata.X[selected_pos, :]
            var = adata.var.copy()

        gene_name_column = next((col for col in GENE_NAME_COLUMNS if col in var.columns), None)
        gene_names = var[gene_name_column].astype(str).tolist() if gene_name_column else var.index.astype(str).tolist()
        counts = matrix.toarray() if issparse(matrix) else np.asarray(matrix)
        counts_df = pd.DataFrame(counts, index=obs.index, columns=gene_names)
        if counts_df.columns.has_duplicates:
            counts_df = counts_df.T.groupby(level=0, sort=False).sum().T

        metadata = obs.reindex(columns=["smoking_status", "age_or_mean_of_age_range", "BMI", "disease"]).copy()
        metadata.index = counts_df.index
        return counts_df, metadata
    finally:
        if getattr(adata, "file", None) is not None:
            adata.file.close()


def score_group(
    adata_path: Path,
    group_name: str,
    *,
    model,
    config: dict,
    max_cells: int | None,
    random_seed: int,
) -> pd.DataFrame:
    counts_df, metadata = load_group_counts_df(
        adata_path,
        counts_source=COUNTS_SOURCE,
        max_cells=max_cells,
        random_seed=random_seed,
    )
    selected_df, to_fill_columns, selected_var = main_gene_selection(counts_df, gene_list)

    counts_19264 = selected_df.to_numpy(dtype=np.float32)
    total_counts = counts_19264.sum(axis=1).astype(np.float32)
    normalized = np.log1p((counts_19264 / np.clip(total_counts[:, None], 1.0, None)) * 1e4).astype(np.float32)

    s_token = np.log10(np.clip(total_counts, 1.0, None)).astype(np.float32)
    seq = np.concatenate([normalized, s_token[:, None], s_token[:, None]], axis=1).astype(np.float32)

    rng = np.random.default_rng(random_seed)
    zero_padded = selected_var["mask"].to_numpy(dtype=bool)
    gene_mask = np.zeros_like(counts_19264, dtype=bool)
    valid_candidates = (counts_19264 > 0) & (~zero_padded[None, :])

    for i in range(counts_19264.shape[0]):
        candidates = np.flatnonzero(valid_candidates[i])
        if candidates.size == 0:
            continue
        n_mask = max(1, int(np.floor(candidates.size * MASK_FRACTION)))
        chosen = candidates if n_mask >= candidates.size else rng.choice(candidates, size=n_mask, replace=False)
        gene_mask[i, chosen] = True

    seq_masked = seq.copy()
    seq_raw = seq.copy()
    seq_masked[:, :19264][gene_mask] = float(config["mask_token_id"])
    seq_raw[:, :19264][gene_mask] = 0.0

    predictions = []
    try:
        with torch.inference_mode():
            for start in range(0, seq_masked.shape[0], BATCH_SIZE):
                stop = min(start + BATCH_SIZE, seq_masked.shape[0])
                x = torch.from_numpy(seq_masked[start:stop]).to(device=DEVICE, dtype=torch.float32)
                x_raw = torch.from_numpy(seq_raw[start:stop]).to(device=DEVICE, dtype=torch.float32)
                (
                    encoder_data,
                    encoder_position_gene_ids,
                    encoder_data_padding,
                    encoder_labels,
                    decoder_data,
                    decoder_data_padding,
                    _,
                    _,
                    decoder_position_gene_ids,
                ) = getEncoerDecoderData(x, x_raw, config)

                batch_prediction = model.forward(
                    x=encoder_data,
                    padding_label=encoder_data_padding,
                    encoder_position_gene_ids=encoder_position_gene_ids,
                    encoder_labels=encoder_labels,
                    decoder_data=decoder_data,
                    mask_gene_name=False,
                    mask_labels=None,
                    decoder_position_gene_ids=decoder_position_gene_ids,
                    decoder_data_padding_labels=decoder_data_padding,
                )
                predictions.append(batch_prediction[:, :19264].detach().cpu())
                clear_gpu_memory(
                    x,
                    x_raw,
                    encoder_data,
                    encoder_position_gene_ids,
                    encoder_data_padding,
                    encoder_labels,
                    decoder_data,
                    decoder_data_padding,
                    decoder_position_gene_ids,
                    batch_prediction,
                )

        prediction = torch.cat(predictions, dim=0).numpy()
    finally:
        del predictions
        clear_gpu_memory()

    masked_gene_count = gene_mask.sum(axis=1)
    masked_mse = ((prediction - normalized) ** 2 * gene_mask).sum(axis=1) / np.clip(masked_gene_count, 1, None)

    results = metadata.copy()
    results["group_name"] = group_name
    results["cell_id"] = selected_df.index.to_numpy()
    results["seq_depth"] = total_counts.astype(np.float64)
    results["log10_seq_depth"] = np.log10(np.clip(total_counts, 1.0, None)).astype(np.float64)
    results["masked_gene_count"] = masked_gene_count.astype(np.int32)
    results["masked_mse"] = masked_mse.astype(np.float64)
    results["predictability"] = -results["masked_mse"]
    results["zero_padded_genes"] = int(len(to_fill_columns))
    results["smoking_status"] = results["smoking_status"].astype("string").str.strip().str.lower()
    results["smoking_code"] = results["smoking_status"].map(SMOKING_CODE)
    results["age_or_mean_of_age_range"] = pd.to_numeric(results["age_or_mean_of_age_range"], errors="coerce")
    results["BMI"] = pd.to_numeric(results["BMI"], errors="coerce")
    return results.reset_index(drop=True)


def build_correlation_table(frame: pd.DataFrame, variables: list[str], response_col: str) -> pd.DataFrame:
    rows = []
    scopes = [("all", frame), *[(name, sub) for name, sub in frame.groupby("group_name")]]
    for scope_name, sub in scopes:
        for variable in variables:
            pair = sub[[variable, response_col]].replace([np.inf, -np.inf], np.nan).dropna()
            if len(pair) < 3 or pair[variable].nunique() < 2 or pair[response_col].nunique() < 2:
                pearson = np.nan
                spearman = np.nan
            else:
                pearson = pair[variable].corr(pair[response_col], method="pearson")
                spearman = pair[variable].corr(pair[response_col], method="spearman")
            rows.append({
                "scope": scope_name,
                "response": response_col,
                "variable": variable,
                "cells_used": int(len(pair)),
                "pearson": float(pearson) if pd.notna(pearson) else np.nan,
                "spearman": float(spearman) if pd.notna(spearman) else np.nan,
            })
    return pd.DataFrame(rows)


In [ ]:
group_frames = []
model = None
config = None

try:
    model, config = load_model_frommmf(str(CKPT_PATH), key="gene", device=DEVICE)
    model.eval()

    for offset, (group_name, group_path) in enumerate(GROUP_PATHS.items()):
        frame = score_group(
            group_path,
            group_name,
            model=model,
            config=config,
            max_cells=MAX_CELLS_PER_GROUP,
            random_seed=RANDOM_SEED + offset,
        )
        group_frames.append(frame)
finally:
    clear_gpu_memory(model)

results_df = pd.concat(group_frames, ignore_index=True)
results_df.head()

In [ ]:
group_summary = results_df.groupby("group_name").agg(
    cells=("cell_id", "size"),
    median_mse=("masked_mse", "median"),
    mean_mse=("masked_mse", "mean"),
    median_seq_depth=("seq_depth", "median"),
    median_age=("age_or_mean_of_age_range", "median"),
    median_bmi=("BMI", "median"),
)

depth_mse_corr = build_correlation_table(results_df, ["log10_seq_depth"], "masked_mse")
risk_predictability_corr = build_correlation_table(
    results_df,
    ["age_or_mean_of_age_range", "BMI", "smoking_code"],
    "predictability",
)
smoking_summary = results_df.groupby("smoking_status").agg(
    cells=("cell_id", "size"),
    median_mse=("masked_mse", "median"),
    mean_mse=("masked_mse", "mean"),
)

display(group_summary)
display(depth_mse_corr)
display(risk_predictability_corr)
display(smoking_summary)

In [ ]:
group_order = list(GROUP_PATHS.keys())
smoking_order = [name for name in ["never", "former", "active"] if name in results_df["smoking_status"].dropna().unique()]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

group_values = [results_df.loc[results_df["group_name"] == group_name, "masked_mse"].dropna().to_numpy() for group_name in group_order]
axes[0].violinplot(group_values, showmeans=True, showmedians=True)
axes[0].set_xticks(np.arange(1, len(group_order) + 1))
axes[0].set_xticklabels(group_order)
axes[0].set_title("Masked MSE by Group")
axes[0].set_ylabel("Masked MSE")

if smoking_order:
    smoking_values = [results_df.loc[results_df["smoking_status"] == smoking_name, "masked_mse"].dropna().to_numpy() for smoking_name in smoking_order]
    axes[1].violinplot(smoking_values, showmeans=True, showmedians=True)
    axes[1].set_xticks(np.arange(1, len(smoking_order) + 1))
    axes[1].set_xticklabels(smoking_order)
    axes[1].set_title("Masked MSE by Smoking Status")
    axes[1].set_ylabel("Masked MSE")
else:
    axes[1].text(0.5, 0.5, "No smoking-status variation in sampled cells", ha="center", va="center")
    axes[1].set_axis_off()

fig.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
colors = {"healthy": "#1f77b4", "risk_group": "#d62728"}
plot_specs = [
    ("log10_seq_depth", "log10 sequencing depth vs masked MSE"),
    ("age_or_mean_of_age_range", "Age vs masked MSE"),
    ("BMI", "BMI vs masked MSE"),
]

for ax, (column, title) in zip(axes, plot_specs):
    for group_name, sub in results_df.groupby("group_name"):
        pair = sub[[column, "masked_mse"]].replace([np.inf, -np.inf], np.nan).dropna()
        if pair.empty:
            continue
        ax.scatter(
            pair[column],
            pair["masked_mse"],
            s=18,
            alpha=0.45,
            label=group_name,
            color=colors.get(group_name),
        )
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.grid(alpha=0.2)

axes[0].set_ylabel("Masked MSE")
axes[0].legend()
fig.tight_layout()
plt.show()